In [1]:
# 6-21-2026

In [2]:
import pandas as pd
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr
import glob
import os

In [17]:
domain_id = "22"

X_train = pd.read_csv(f"train_X/domain_{domain_id}.csv")
y_train = pd.read_csv(f"train_y/domain_{domain_id}.csv")["log_ba"]
X_test = pd.read_csv(f"test_X/domain_{domain_id}.csv")
y_test = pd.read_csv(f"test_y/domain_{domain_id}.csv")["log_ba"]

scaler = joblib.load(f"scalers/domain_{domain_id}.joblib")
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [18]:
# val split only for comparing hyperparam choices, not used in the final loop. rf doesn't need it for fitting itself
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=5
)

In [19]:
model = RandomForestRegressor(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=5,
    max_features="sqrt",
    n_jobs=-1,
    random_state=5
)
model.fit(X_tr, y_tr)

,n_estimators,600
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [20]:
val_pred = model.predict(X_val)
test_pred = model.predict(X_test_scaled)

val_spearman, _ = spearmanr(y_val, val_pred)
test_spearman, _ = spearmanr(y_test, test_pred)
val_r2 = r2_score(y_val, val_pred)
test_r2 = r2_score(y_test, test_pred)

In [21]:
print(f"val spearman: {val_spearman:.4f}, test spearman: {test_spearman:.4f}")
print(f"val r2: {val_r2:.4f}, test r2: {test_r2:.4f}")

val spearman: 0.4584, test spearman: 0.4499
val r2: 0.1314, test r2: 0.1384


In [3]:
RF_PARAMS = {
    "n_estimators": 600,
    "max_depth": None,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "n_jobs": -1,
    "random_state": 10
}

In [4]:
domain_files = glob.glob("train_X/domain_*.csv")
domain_ids = sorted(
    int(os.path.basename(f).replace("domain_", "").replace(".csv", ""))
    for f in domain_files
)
print(f"found {len(domain_ids)} domains")

found 34 domains


In [5]:
models = {}
scalers = {}

In [6]:
# train rf models
for domain_id in domain_ids:
    X_train = pd.read_csv(f"train_X/domain_{domain_id}.csv")
    y_train = pd.read_csv(f"train_y/domain_{domain_id}.csv")["log_ba"] # only series

    scaler = joblib.load(f"scalers/domain_{domain_id}.joblib")
    X_train_scaled = scaler.transform(X_train)

    model = RandomForestRegressor(**RF_PARAMS)
    model.fit(X_train_scaled, y_train)

    models[domain_id] = model
    scalers[domain_id] = scaler

    print(f"domain {domain_id} done")
# takes ~50 min

domain 0 done
domain 1 done
domain 2 done
domain 4 done
domain 5 done
domain 6 done
domain 7 done
domain 8 done
domain 11 done
domain 12 done
domain 13 done
domain 16 done
domain 18 done
domain 19 done
domain 20 done
domain 21 done
domain 22 done
domain 23 done
domain 25 done
domain 26 done
domain 27 done
domain 28 done
domain 29 done
domain 30 done
domain 32 done
domain 33 done
domain 36 done
domain 37 done
domain 38 done
domain 39 done
domain 45 done
domain 46 done
domain 47 done
domain 49 done


In [7]:
test_X_raw = {}
test_y = {}
# load up the testing datasets
for domain_id in domain_ids:
    test_X_raw[domain_id] = pd.read_csv(f"test_X/domain_{domain_id}.csv")
    test_y[domain_id] = pd.read_csv(f"test_y/domain_{domain_id}.csv")["log_ba"]

In [8]:
T_spearman_rf = pd.DataFrame(index=domain_ids, columns=domain_ids, dtype=float)

In [9]:
for i in domain_ids:
    model_i = models[i]
    scaler_i = scalers[i]

    for j in domain_ids:
        # apply source domain's scaler to target domain's raw test X, not target's own scaler
        X_test_scaled = scaler_i.transform(test_X_raw[j])
        y_true = test_y[j]

        preds = model_i.predict(X_test_scaled)
        T_spearman_rf.loc[i, j], _ = spearmanr(y_true, preds)

    print(f"finished evaluating source domain {i} against all targets")
    # takes ~70 min

finished evaluating source domain 0 against all targets
finished evaluating source domain 1 against all targets
finished evaluating source domain 2 against all targets
finished evaluating source domain 4 against all targets
finished evaluating source domain 5 against all targets
finished evaluating source domain 6 against all targets
finished evaluating source domain 7 against all targets
finished evaluating source domain 8 against all targets
finished evaluating source domain 11 against all targets
finished evaluating source domain 12 against all targets
finished evaluating source domain 13 against all targets
finished evaluating source domain 16 against all targets
finished evaluating source domain 18 against all targets
finished evaluating source domain 19 against all targets
finished evaluating source domain 20 against all targets
finished evaluating source domain 21 against all targets
finished evaluating source domain 22 against all targets
finished evaluating source domain 23 ag

In [10]:
T_spearman_rf

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.392599,0.292470,0.126048,0.063233,0.215725,0.081519,0.145306,0.181991,0.172674,0.025168,...,0.230056,0.245248,0.150715,0.277713,0.113371,-0.009352,0.116259,0.154295,0.022475,0.026572
1,0.144293,0.395763,0.011847,0.013593,0.177290,0.062480,0.048581,0.122780,0.091267,0.048519,...,0.161339,0.021949,0.028618,0.192355,0.042696,0.066730,0.102649,0.198859,0.034054,-0.041965
2,0.195723,0.212534,0.367785,0.025322,0.203882,0.134139,0.179099,0.228553,0.157355,0.054087,...,0.194900,0.228476,0.120756,0.243839,0.139353,0.055978,-0.002842,0.247087,0.038750,0.055952
4,0.237381,0.231039,0.126367,0.410439,0.214871,0.189061,0.132013,0.190776,0.160916,0.032841,...,0.174782,0.247686,0.147801,0.283826,0.220441,-0.003888,0.022144,0.308351,0.135801,0.013121
5,0.190374,0.205407,0.101130,0.087768,0.464813,0.143357,0.164309,0.208233,0.267519,0.103724,...,0.195042,0.231232,0.118464,0.290430,0.172006,-0.016249,0.115628,0.295632,0.067739,0.046587
6,0.173769,0.235766,0.029831,0.218848,0.192115,0.316842,0.106066,0.162303,0.112873,0.027397,...,0.204713,0.249182,0.124091,0.255471,0.198435,-0.071297,0.096571,0.321126,0.098624,-0.024959
7,0.119728,0.071243,0.059731,0.071864,0.059897,0.108947,0.330633,0.101696,0.170778,0.051790,...,0.185503,0.181481,0.146499,0.208720,0.207935,0.038649,0.043701,0.164180,-0.000444,0.022502
8,0.181936,0.227220,-0.005685,0.026613,0.135382,0.107438,0.197567,0.424987,0.063023,0.000075,...,0.087558,0.141930,0.101962,0.173548,0.082901,-0.001850,0.052477,0.220530,0.051328,0.013053
11,0.214186,0.227551,0.110486,0.129279,0.236645,0.082140,0.101353,0.209500,0.551058,-0.007034,...,0.256736,0.162890,0.103278,0.366901,0.183326,-0.010493,0.148702,0.210893,-0.056373,0.068730
12,0.121809,0.189351,-0.016999,0.115648,0.161226,0.068627,0.073154,0.017381,0.100046,0.502653,...,0.167424,-0.108972,0.047022,0.172082,0.117883,0.077839,0.088635,0.127615,0.067925,-0.070449


In [11]:
T_spearman_rf.to_csv("transfer_matrix_spearman_rf_10.csv")